# 사용자실험 로그 분석

이 노트북은 사용자실험 로그를 **데이터 준비 → 로그 정제 → 행동지표 계산 → 3요인 분석 → 민감도 분석 → 논문용 표 → 성공률 분석** 순서로 분석합니다.

각 단계는 먼저 목적을 설명하고, 바로 아래의 짧은 코드 셀에서 한 가지 작업만 수행합니다. 위에서 아래로 차례대로 실행하면 전체 분석 흐름을 따라갈 수 있습니다.


별표 기준: ** p<.05, * .05<p<.10. p=.05와 p=.10에는 별표를 붙이지 않으며, 반올림 전 p값으로 판정합니다.


## 0. 분석 설계 한눈에 보기

| 구분 | 정의 |
|---|---|
| UI | A2UI / Baseline |
| 숙련도 | 초보 / 숙련 |
| Task | Task 1 / Task 2 |
| Number of Prompts | 각 Task에서 사용한 실제 프롬프트 수 |
| 수행시간 | 각 Task의 첫 요청 시작부터 마지막 로그까지의 실제 시간(분) |
| 반복측정 | 한 참가자가 Task 1과 Task 2를 모두 수행 |
| 3요인 모형 1 | `Number of Prompts ~ UI × 숙련도 × Task` |
| 3요인 모형 2 | `수행시간 ~ UI × 숙련도 × Task` |
| 분석에서 제외 | 과제 순서(Task Order), 30분 이내 완료 여부, 입력 글자 수 |

숙련도는 사전 AI 리터러시 평균이 6.0 이상이면 `숙련`, 6.0 미만이면 `초보`로 구분합니다. A2UI의 `confirmed_execution`은 기존 질문을 확인하는 단계이므로 새로운 프롬프트로 세지 않습니다.


## 1. 분석 환경과 참가자 정보 준비

먼저 분석에 필요한 라이브러리와 파일 위치를 준비한 뒤, 설문 파일에서 참가자의 UI 조건과 숙련도를 만듭니다.


### 1-1. 라이브러리 불러오기

데이터 처리에는 pandas, 통계 분석에는 statsmodels와 scipy, 그림에는 matplotlib을 사용합니다.


In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from IPython.display import Image, Markdown, display

KOREAN_FONT = common.configure_plots()


### 1-2. 입력 파일 위치 정하기

프로젝트 루트와 `Data/A2UI` 폴더 중 어디에서 실행해도 같은 파일을 찾도록 경로를 설정합니다.


In [ ]:
A2UI_DIR = common.A2UI_DIR
LOG_DIR = A2UI_DIR.parent / "logs"
ANALYSIS_DIR = A2UI_DIR.parent / "analysis"

SURVEY_FILE = A2UI_DIR / "merged_survey.xlsx"
A2UI_LOG_FILE = LOG_DIR / "4_3_user_agent2ui_interactions_annotated.csv"
BASELINE_LOG_FILE = LOG_DIR / "4_3_user_baseline_interactions_annotated.csv"
FAILURE_FILE = ANALYSIS_DIR / "participant_final_failure_reclassification.xlsx"
SURVEY_FINAL_FILE = common.RESULT_DIR / "survey_final_anova.csv"
SURVEY_MANIFEST = common.RESULT_DIR / "survey_final_manifest.json"


### 1-3. 입력 파일 존재 여부 확인하기

파일이 하나라도 없으면 이후 결과가 잘못될 수 있으므로 분석 시작 전에 즉시 중단합니다.


In [ ]:
INPUT_FILES = [
    SURVEY_FILE, A2UI_LOG_FILE, BASELINE_LOG_FILE,
    FAILURE_FILE, SURVEY_FINAL_FILE, SURVEY_MANIFEST,
]

for path in INPUT_FILES:
    assert path.exists(), f"파일을 찾을 수 없습니다: {path}"

print(f"입력 파일 {len(INPUT_FILES)}개를 모두 확인했습니다.")


### 1-4. 참가자 기본정보 불러오기

분석에 필요한 참가자 ID, 인터페이스 조건, 사전 AI 리터러시 점수만 선택합니다.


In [ ]:
survey = common.load_survey()
participants = survey[["참여자번호"]].rename(columns={"참여자번호": "participant_id"}).join(common.make_design(survey))
participants["숙련도"] = participants["숙련도"].replace({"초보자": "초보", "숙련자": "숙련"})
print(f"참가자 수: {len(participants)}명")


### 1-6. 집단별 참가자 수 확인하기

3요인 분석 전에 UI와 숙련도 조합별 표본 수를 확인합니다.


In [ ]:
group_count = pd.crosstab(
    participants["UI"], participants["숙련도"]
).reindex(index=["A2UI", "Baseline"], columns=["초보", "숙련"])

group_count["전체"] = group_count.sum(axis=1)
group_count.loc["전체"] = group_count.sum(axis=0)
display(group_count.rename_axis("UI 유형"))


**해석:** A2UI와 Baseline은 각각 20명이고, 초보와 숙련도 각각 20명입니다. UI × 숙련도 세부 집단은 각각 10명입니다. 분석에서는 단순 평균 비교보다 UI, 숙련도, Task와 그 상호작용을 한 모형에서 함께 검정합니다.


## 2. 로그 정제와 Task 구분

이 단계에서는 분석에 사용할 로그만 남기고, 각 로그가 Task 1과 Task 2 중 어디에 속하는지 정합니다.

재실험 참가자는 최종 재실험 기록만 사용합니다. 사후 보정 스레드와 `summary` 행은 수행시간과 프롬프트 수 계산에서 제외합니다.


### 2-1. A2UI와 Baseline 로그 합치기

두 UI 조건의 로그를 같은 열 구조로 세로 결합합니다.


In [ ]:
a2ui = pd.read_csv(A2UI_LOG_FILE, low_memory=False)
baseline = pd.read_csv(BASELINE_LOG_FILE, low_memory=False)
raw_logs = pd.concat([a2ui, baseline], ignore_index=True)

print(f"A2UI {len(a2ui)}행 + Baseline {len(baseline)}행 = {len(raw_logs)}행")


### 2-2. 최종 실험 기록만 선택하기

재실험이 있는 참가자는 최종 재실험 로그만 사용하고, 사후 보정용 replay 스레드와 summary 행은 제외합니다.


In [ ]:
RERUN_ORIGIN = "observed_rerun_20260830"
# Public source loads study-specific exclusions from a private local config.
REANALYSIS_THREADS = set(common.REANALYSIS_THREADS)

rerun_ids = set(raw_logs.loc[
    raw_logs["analysis_record_origin"].eq(RERUN_ORIGIN),
    "participant_id",
])


In [ ]:
use_final_run = (
    ~raw_logs["participant_id"].isin(rerun_ids)
    | raw_logs["analysis_record_origin"].eq(RERUN_ORIGIN)
)
reanalysis_mask = raw_logs["thread_id"].isin(REANALYSIS_THREADS)
summary_mask = raw_logs["stage"].eq("summary")

analysis_mask = use_final_run & ~reanalysis_mask & ~summary_mask
logs = raw_logs.loc[analysis_mask].copy()


### 2-3. 로그 시각을 같은 형식으로 바꾸기

서로 다른 문자열 시각 형식을 UTC datetime으로 통일합니다. 각 로그의 시작 시각은 종료 시각에서 처리시간을 빼서 계산합니다.


In [ ]:
clean_time = (
    logs["created_at"].astype(str)
    .str.replace(r"^[A-Za-z]{3} ", "", regex=True)
    .str.replace(r" \(Korean Standard Time\)$", "", regex=True)
)
logs["timestamp"] = pd.to_datetime(
    clean_time, format="mixed", utc=True, errors="coerce"
)
logs["duration_ms"] = pd.to_numeric(
    logs["duration_ms"], errors="coerce"
).fillna(0)
logs["event_start"] = logs["timestamp"] - pd.to_timedelta(
    logs["duration_ms"], unit="ms"
)


### 2-4. 참가자 조건을 로그에 연결하기

각 로그 행에 참가자의 UI와 숙련도를 붙이고, 참가자와 시간 순서로 정렬합니다.


In [ ]:
logs = logs.merge(
    participants[["participant_id", "UI", "숙련도"]],
    on="participant_id", how="left", validate="many_to_one",
)
logs = logs.sort_values(
    ["participant_id", "timestamp", "thread_id"]
).reset_index(drop=True)

assert logs[["timestamp", "UI", "숙련도"]].notna().all().all()


### 2-5. Task 시작 질문을 찾는 규칙 만들기

- `60일`이 처음 등장하면 Task 1 시작
- `90일`, `고객`(또는 `고개` 오타), `주문 수`가 함께 처음 등장하면 Task 2 시작

질문마다 Task를 새로 추측하지 않고, 참가자별 최초 시작점만 경계로 사용합니다.


In [ ]:
def detect_task_start(question):
    text = re.sub(r"\s+", "", str(question))
    if "60일" in text:
        return "Task 1"
    if (
        "90일" in text and "주문수" in text
        and any(word in text for word in ["고객", "고개"])
    ):
        return "Task 2"
    return pd.NA


### 2-6. 각 로그 행에 Task 붙이기

참가자별 Task 시작점을 찾은 뒤 다음 Task가 시작될 때까지 같은 Task 값을 아래로 채웁니다. 공식 과제 시작 전 행은 분석에서 제외합니다.


In [ ]:
logs["task_candidate"] = logs["question"].apply(detect_task_start)
candidate = logs["task_candidate"].notna()

first_start = ~logs.loc[
    candidate, ["participant_id", "task_candidate"]
].duplicated()
logs["task_start"] = pd.NA
logs.loc[candidate, "task_start"] = logs.loc[
    candidate, "task_candidate"
].where(first_start.to_numpy())
logs["Task"] = logs.groupby("participant_id")["task_start"].ffill()

pre_task_rows = int(logs["Task"].isna().sum())
logs = logs.loc[logs["Task"].notna()].copy()


### 2-7. 실제 프롬프트 행 표시하기

질문이 비어 있지 않으면서 `confirmed_execution`이 아닌 행만 실제 프롬프트로 셉니다.


In [ ]:
has_question = logs["question"].fillna("").str.strip().ne("")
logs["is_prompt"] = (
    has_question & logs["stage"].ne("confirmed_execution")
)

start_check = pd.crosstab(logs["participant_id"], logs["task_start"])
assert (start_check[["Task 1", "Task 2"]].eq(1)).all().all()
assert not logs.loc[
    logs["stage"].eq("confirmed_execution"), "is_prompt"
].any()


### 2-8. 정제 결과 점검하기

각 제외 규칙이 몇 행에 적용됐는지 표로 남겨 재현성을 확인합니다.


In [ ]:
cleaning_audit = pd.Series({
    "통합 원로그 행": len(raw_logs),
    "재실험 대체로 제외": int((~use_final_run).sum()),
    "사후 보정 스레드 제외": int((use_final_run & reanalysis_mask).sum()),
    "summary 행 제외": int((use_final_run & ~reanalysis_mask & summary_mask).sum()),
    "공식 과제 시작 전 제외": pre_task_rows,
    "행동분석 사용 행": len(logs),
    "분석 참가자": logs["participant_id"].nunique(),
    "Task 시작점": int(logs["task_start"].notna().sum()),
    "최종 재실험 참가자": ", ".join(sorted(rerun_ids)),
})
display(cleaning_audit.to_frame("값"))


**정제 결과 확인:** 위 감사표에서 원로그·제외 행·최종 분석 행·참가자·Task 시작점 수를 확인합니다.

## 3. 수행시간과 Number of Prompts 계산

분석 단위는 **참가자 × Task**입니다. 한 참가자에게 최종적으로 Task 1 한 행과 Task 2 한 행이 생기므로 40명 × 2개 Task = 80행이 됩니다.


### 3-1. 스레드별 행동지표 계산하기

같은 Task가 여러 스레드에 걸쳐 있을 수 있으므로 먼저 스레드 단위로 시작·종료 시각과 프롬프트 수를 계산합니다.


In [ ]:
session = (
    logs.groupby(
        ["participant_id", "UI", "숙련도", "Task", "thread_id"],
        as_index=False,
    )
    .agg(
        start=("event_start", "min"),
        end=("timestamp", "max"),
        Number_of_Prompts=("is_prompt", "sum"),
    )
)
session["수행시간_분"] = (
    session["end"] - session["start"]
).dt.total_seconds() / 60


### 3-2. 참가자 × Task 단위로 합치기

같은 Task의 여러 스레드를 합산합니다. 이렇게 하면 스레드 사이의 긴 휴지시간은 수행시간에 포함되지 않습니다.


In [ ]:
behavior = (
    session.groupby(
        ["participant_id", "UI", "숙련도", "Task"],
        as_index=False,
    )
    .agg(
        수행시간_분=("수행시간_분", "sum"),
        Number_of_Prompts=("Number_of_Prompts", "sum"),
    )
)

assert len(behavior) == 80
assert behavior[["수행시간_분", "Number_of_Prompts"]].ge(0).all().all()
display(behavior.head())


### 3-3. 참가자 전체 합계 만들기

UI별 전체 기술통계를 보기 위해 각 참가자의 Task 1과 Task 2 값을 합산합니다. 3요인 검정에는 이 합계가 아니라 앞의 80행짜리 `behavior`를 사용합니다.


In [ ]:
participant_behavior = (
    behavior.groupby(
        ["participant_id", "UI", "숙련도"],
        as_index=False,
    )
    .agg(
        수행시간_분=("수행시간_분", "sum"),
        Number_of_Prompts=("Number_of_Prompts", "sum"),
    )
)
assert len(participant_behavior) == 40


### 3-4. UI와 숙련도별 전체 기술통계

아래 집단별로 참가자 전체 수행시간과 프롬프트 수의 평균(M)과 표준편차(SD)를 계산합니다.


In [ ]:
groups = [
    ("A2UI", participant_behavior["UI"].eq("A2UI")),
    ("Baseline", participant_behavior["UI"].eq("Baseline")),
    ("초보", participant_behavior["숙련도"].eq("초보")),
    ("숙련", participant_behavior["숙련도"].eq("숙련")),
    ("A2UI × 초보", participant_behavior["UI"].eq("A2UI") & participant_behavior["숙련도"].eq("초보")),
    ("A2UI × 숙련", participant_behavior["UI"].eq("A2UI") & participant_behavior["숙련도"].eq("숙련")),
    ("Baseline × 초보", participant_behavior["UI"].eq("Baseline") & participant_behavior["숙련도"].eq("초보")),
    ("Baseline × 숙련", participant_behavior["UI"].eq("Baseline") & participant_behavior["숙련도"].eq("숙련")),
]


In [ ]:
overall_descriptives = pd.DataFrame([
    {
        "구분": label,
        "n": int(mask.sum()),
        "수행시간 M": participant_behavior.loc[mask, "수행시간_분"].mean(),
        "수행시간 SD": participant_behavior.loc[mask, "수행시간_분"].std(),
        "Prompts M": participant_behavior.loc[mask, "Number_of_Prompts"].mean(),
        "Prompts SD": participant_behavior.loc[mask, "Number_of_Prompts"].std(),
    }
    for label, mask in groups
]).set_index("구분")

display(overall_descriptives.round(2))


**기술통계 해석:** 위 표의 UI·숙련도별 프롬프트 수와 수행시간을 비교합니다. 평균 차이의 통계적 판단은 4절의 3요인 분석을 기준으로 합니다.

### 3-5. UI별 Number of Prompts 기술통계

논문에서 분포를 함께 설명할 수 있도록 평균, 표준편차, 중앙값, 최솟값, 최댓값을 정리합니다. 이 표는 두 Task의 합계 기준입니다.


In [ ]:
prompt_descriptives = (
    participant_behavior.groupby("UI", observed=True)["Number_of_Prompts"]
    .agg(N="count", M="mean", SD="std", Median="median", Min="min", Max="max")
    .reindex(["A2UI", "Baseline"])
)
prompt_descriptives.index.name = "UI 유형"

prompt_table = prompt_descriptives.copy()
prompt_table[["M", "SD", "Median"]] = prompt_table[["M", "SD", "Median"]].round(2)
prompt_table[["N", "Min", "Max"]] = prompt_table[["N", "Min", "Max"]].astype(int)
display(prompt_table)


### 3-6. UI별 프롬프트 수 분포

상자그림·참가자 점·그림 저장을 한 셀에서 수행합니다. 난수 시드를 고정해 점 위치를 재현합니다.

In [ ]:
plot_order = ["Baseline", "A2UI"]
plot_colors = ["#4C8EDA", "#F07B4F"]
plot_values = [
    participant_behavior.loc[
        participant_behavior["UI"].eq(ui),
        "Number_of_Prompts",
    ].to_numpy()
    for ui in plot_order
]
rng = np.random.default_rng(20260914)

with plt.rc_context({"font.family": KOREAN_FONT, "axes.unicode_minus": False}):
    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    boxes = ax.boxplot(
        plot_values, tick_labels=plot_order, widths=0.42,
        patch_artist=True, showfliers=False,
        medianprops={"color": "black", "linewidth": 1.8},
        whiskerprops={"color": "#555555", "linewidth": 1.2},
        capprops={"color": "#555555", "linewidth": 1.2},
    )
    for patch, color in zip(boxes["boxes"], plot_colors):
        patch.set(facecolor=color, edgecolor=color, alpha=0.28, linewidth=1.3)
    plt.close(fig)  # 점을 추가하기 전 중간 그림은 출력하지 않음

for position, values, color in zip(range(1, 3), plot_values, plot_colors):
    jitter = rng.uniform(-0.11, 0.11, size=len(values))
    ax.scatter(
        position + jitter, values, s=34, color=color,
        edgecolor="white", linewidth=0.55, alpha=0.95, zorder=3,
    )
ax.set_ylabel("프롬프트 수")
ax.set_xlabel("")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#D9D5CE", linewidth=0.8, alpha=0.75)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

FIGURE_DIR = A2UI_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

prompt_figure_png = FIGURE_DIR / "figure_05_prompt_distribution.png"
prompt_figure_pdf = FIGURE_DIR / "figure_05_prompt_distribution.pdf"
fig.savefig(prompt_figure_png, dpi=300, bbox_inches="tight")
fig.savefig(prompt_figure_pdf, bbox_inches="tight")
display(Image(filename=prompt_figure_png))

display(Markdown(
    "*그림 5. UI 조건별 프롬프트 수 분포 (개별 참가자 점 포함, N = 40).*"
))


### 3-9. UI × 숙련도 × Task 셀별 기술통계

3요인 모형의 여덟 조건 조합별로 평균과 표준편차를 확인합니다.


In [ ]:
task_descriptives = (
    behavior.groupby(["UI", "숙련도", "Task"], observed=True)
    .agg(
        n=("participant_id", "nunique"),
        수행시간_M=("수행시간_분", "mean"),
        수행시간_SD=("수행시간_분", "std"),
        Prompts_M=("Number_of_Prompts", "mean"),
        Prompts_SD=("Number_of_Prompts", "std"),
    )
)


In [ ]:
CELL_ORDER = pd.MultiIndex.from_product(
    [
        ["A2UI", "Baseline"],
        ["초보", "숙련"],
        ["Task 1", "Task 2"],
    ],
    names=["UI", "숙련도", "Task"],
)
task_descriptives = task_descriptives.reindex(CELL_ORDER)
display(task_descriptives.round(2))


**과제별 비교:** 위 표에서 각 UI × 숙련도 집단의 Task 1·2 평균과 표준편차를 확인합니다. 과제 차이가 집단에 따라 달라지는지는 다음 절의 상호작용 검정으로 판단합니다.

## 4. UI × 숙련도 × Task 3요인 분석

### 4-1. 모형의 의미

두 종속변수의 분석식은 다음과 같습니다.

`Number of Prompts ~ UI × 숙련도 × Task`

`수행시간 ~ UI × 숙련도 × Task`

`×`는 세 주효과뿐 아니라 2요인 상호작용 3개와 3요인 상호작용 1개를 모두 포함한다는 뜻입니다. Task는 같은 참가자 안에서 반복되므로 참가자 ID로 군집화한 강건 표준오차와 소표본 보정 F 검정을 사용합니다. 과제 순서는 모형에 포함하지 않습니다.


In [ ]:
factor_structure = pd.DataFrame({
    "요인": ["UI", "숙련도", "Task"],
    "수준": ["A2UI / Baseline", "초보 / 숙련", "Task 1 / Task 2"],
    "역할": ["참가자 간", "참가자 간", "참가자 내 반복측정"],
})
display(factor_structure.set_index("요인"))


### 4-2. 분석식과 효과 이름 정의하기

`Sum` 코딩을 사용하면 주효과가 다른 요인의 수준 전체에 대한 평균 효과로 해석됩니다.


In [ ]:
FORMULA = (
    "{metric} ~ "
    "C(UI, Sum) * C(숙련도, Sum) * C(Task, Sum)"
)

TERM_NAMES = {
    "C(UI, Sum)": "UI",
    "C(숙련도, Sum)": "숙련도",
    "C(Task, Sum)": "Task",
    "C(UI, Sum):C(숙련도, Sum)": "UI × 숙련도",
    "C(UI, Sum):C(Task, Sum)": "UI × Task",
    "C(숙련도, Sum):C(Task, Sum)": "숙련도 × Task",
    "C(UI, Sum):C(숙련도, Sum):C(Task, Sum)": "UI × 숙련도 × Task",
}
EFFECT_ORDER = list(TERM_NAMES.values())


### 4-3. 모형 적합 함수 만들기

같은 분석 절차를 Number of Prompts와 수행시간, 민감도 자료에 반복 적용할 수 있도록 함수로 만듭니다.


In [ ]:
def fit_effect_model(metric, data):
    model = smf.ols(
        FORMULA.format(metric=metric), data=data
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data["participant_id"],
            "use_correction": True,
        },
        use_t=True,
    )
    tests = model.wald_test_terms(scalar=True).table.reset_index()
    return model, tests


### 4-4. 통계 프로그램의 항 이름을 읽기 쉬운 이름으로 바꾸기


In [ ]:
def tidy_effect_tests(raw_tests):
    tests = raw_tests.rename(columns={
        "index": "효과",
        "statistic": "F",
        "pvalue": "p",
        "df_constraint": "df1",
        "df_denom": "df2",
    })
    tests = tests.loc[tests["효과"].isin(TERM_NAMES)].copy()
    tests["효과"] = tests["효과"].map(TERM_NAMES)
    return tests.set_index("효과").reindex(EFFECT_ORDER).reset_index()


### 4-5. Number of Prompts 모형 실행하기

첫 번째 종속변수인 Number of Prompts에 대한 3요인 분석입니다.


In [ ]:
prompt_model, prompt_raw_tests = fit_effect_model(
    "Number_of_Prompts", behavior
)
prompt_tests = tidy_effect_tests(prompt_raw_tests)

assert prompt_tests["효과"].tolist() == EFFECT_ORDER
display(prompt_tests)


### 4-6. 기존 수행시간 모형도 같은 기준으로 실행하기

두 번째 종속변수인 수행시간에도 Number of Prompts와 같은 3요인 모형을 적용합니다. 과제 순서는 포함하지 않습니다.


In [ ]:
duration_model, duration_raw_tests = fit_effect_model(
    'Q("수행시간_분")', behavior
)
duration_tests = tidy_effect_tests(duration_raw_tests)


### 4-7. 두 행동지표의 F 검정 결과 비교하기

`df`는 분자 자유도와 분모 자유도를 `1, 39`처럼 함께 표시합니다.


In [ ]:
def format_effect_table(tests):
    table = tests[["효과", "F", "df1", "df2", "p"]].copy()
    table["df"] = table.apply(
        lambda row: f"{int(row['df1'])}, {int(row['df2'])}", axis=1
    )
    table["F"] = table["F"].round(3)
    table["p"] = table["p"].map(
        lambda value: ("<.001" if value < .001 else f"{value:.3f}") + significance_stars(value)
    )
    return table[["효과", "F", "df", "p"]]


In [ ]:
# Cache formatted results for the final overview.
duration_result_table = format_effect_table(duration_tests).set_index("효과")
prompt_result_table = format_effect_table(prompt_tests).set_index("효과")


**효과 검정 해석:** 검정 결과는 6-2절의 최종 행동 로그 표에서 확인합니다. p<.05는 **, .05<p<.10은 *로 표시합니다. 수치는 이번 실행에서 계산한 값을 사용합니다.

## 5. 예외 참가자 제외 민감도 분석

재실험 참가자와 `exempt` 시스템 오류 참가자 5명을 제외해도 결론이 유지되는지 확인합니다. 민감도 분석은 본 분석을 대체하지 않고, 일부 사례가 결론을 좌우하는지만 점검합니다.


### 5-1. 민감도 분석에서 제외할 참가자 찾기


In [ ]:
exempt_ids = set(raw_logs.loc[
    raw_logs["analysis_success_basis"]
    .fillna("").str.contains("exempt"),
    "participant_id",
])
special_ids = rerun_ids | exempt_ids
sensitivity = behavior.loc[
    ~behavior["participant_id"].isin(special_ids)
].copy()

print(f"제외 참가자: {', '.join(sorted(special_ids))}")
print(f"민감도 분석 참가자 수: {sensitivity['participant_id'].nunique()}명")


### 5-2. 예외 제외 자료에 같은 모형 적용하기


In [ ]:
_, prompt_sensitivity_raw = fit_effect_model(
    "Number_of_Prompts", sensitivity
)
prompt_sensitivity = tidy_effect_tests(prompt_sensitivity_raw)

_, duration_sensitivity_raw = fit_effect_model(
    'Q("수행시간_분")', sensitivity
)
duration_sensitivity = tidy_effect_tests(duration_sensitivity_raw)


### 5-3. 본 분석과 민감도 분석의 p값 비교하기


In [ ]:
def compare_p_values(primary, sensitivity_result, label):
    comparison = primary[["효과", "p"]].merge(
        sensitivity_result[["효과", "p"]], on="효과",
        suffixes=("_본", "_예외제외"),
    )
    return comparison.assign(지표=label)

sensitivity_table = pd.concat([
    compare_p_values(duration_tests, duration_sensitivity, "수행시간"),
    compare_p_values(prompt_tests, prompt_sensitivity, "Number of Prompts"),
], ignore_index=True)


In [ ]:
sensitivity_display = sensitivity_table.rename(columns={
    "p_본": "본 분석 p",
    "p_예외제외": "예외 제외 p",
})
for column in ["본 분석 p", "예외 제외 p"]:
    sensitivity_display[column] = sensitivity_display[column].map(
        lambda value: ("<.001" if value < .001 else f"{value:.3f}") + significance_stars(value)
    )

display(sensitivity_display[
    ["지표", "효과", "본 분석 p", "예외 제외 p"]
])


**민감도 분석 해석:** 위 표의 본 분석 p값과 예외 참가자 제외 p값을 비교해 결론이 유지되는지 확인합니다.

## 6. 논문에 사용할 최종 결과표

6-1은 04-2번 노트북에 저장된 설문 ANOVA 결과를 불러오고, 6-2는 이 노트북에서 계산한 수행시간과 Number of Prompts의 3요인 분석을 하나의 F, df, p 표로 정리합니다.


### 6-4. 설문 ANOVA 최종 표 표시하기


In [ ]:
import json
manifest = json.loads(SURVEY_MANIFEST.read_text(encoding="utf-8"))
assert manifest["survey_sha256"] == common.survey_hash(), "설문이 변경되었습니다. 04-2번을 먼저 실행하세요."
assert manifest["common_sha256"] == common.module_hash(), "공통 코드가 변경되었습니다. 04-2번을 먼저 실행하세요."
survey_final = pd.read_csv(SURVEY_FINAL_FILE)
survey_result_table = survey_final.copy()
survey_result_table["F"] = survey_result_table["F"].map(lambda v: "—" if pd.isna(v) else f"{v:.3f}")
survey_result_table["p"] = survey_result_table["p"].map(format_p)
display(Markdown("### 6-1. 설문 최종 결과 (04-2 원자료·변환 후 채택 결과)"))
display(survey_result_table.set_index(["종속변수", "효과"])[["최종 분석 방법", "F", "df1", "df2", "p"]])


### 6-5. 수행시간과 Number of Prompts 논문용 표 만들기

두 종속변수에 대해 요청한 7개 효과를 `F`, `df`, `p` 순서로 함께 정리합니다. 이 표의 df 표기 `1, 39`는 분자 자유도 1과 분모 자유도 39를 뜻합니다.


In [ ]:
behavior_result_table = pd.concat({
    "수행시간": duration_result_table.reindex(EFFECT_ORDER),
    "Number of Prompts": prompt_result_table.reindex(EFFECT_ORDER),
}, axis=1)

display(Markdown("### 6-2. 행동 로그 3요인 분석 결과"))
display(behavior_result_table)
print(
    "주. 참가자 ID로 군집화한 강건 표준오차와 소표본 보정 "
    "F 검정을 사용했으며, 과제 순서는 모형에서 제외했다."
)


**논문용 해석:** UI × 숙련도 × Task 3요인 분석 결과, Number of Prompts에서 Task의 주효과가 유의했다, F(1, 39)=19.361, p<.001. 수행시간에서도 Task의 주효과가 유의했다, F(1, 39)=32.246, p<.001. 반면 두 종속변수 모두 UI, 숙련도, UI × 숙련도, UI × Task, 숙련도 × Task 및 UI × 숙련도 × Task 효과는 유의하지 않았다. 따라서 Task에 따라 프롬프트 사용량과 수행시간은 달랐지만, 그 차이가 UI나 숙련도에 따라 달라진다는 증거는 확인되지 않았다.


## 7. 전체 과제 최종 Success / Failure

성공 여부는 참가자 단위로 판정합니다. Task 1과 Task 2를 모두 완료한 경우만 Success이며, 하나라도 완료하지 못하면 Failure입니다. 이 결과는 Number of Prompts 3요인 분석과 별도입니다.


### 7-1. 최종 실패 재분류표 읽기


In [ ]:
failure_data = pd.read_excel(
    FAILURE_FILE, sheet_name="최종 실패 참가자", skiprows=3
)
failure_ids = set(failure_data.loc[
    failure_data["참가자"].astype(str).str.fullmatch(r"P\d{3}", na=False),
    "참가자",
])

participant_ids = set(participants["participant_id"])
assert failure_ids <= participant_ids


### 7-2. 모든 참가자에게 두 Task가 있는지 확인하기


In [ ]:
task_coverage = behavior.groupby("participant_id")["Task"].nunique()
coverage_check = task_coverage.reindex(participants["participant_id"])
assert coverage_check.eq(2).all(), "두 과제 로그가 모두 있는지 확인하세요."

print(f"두 Task가 모두 있는 참가자: {coverage_check.eq(2).sum()}명")


### 7-3. 참가자별 최종 결과 만들기


In [ ]:
outcomes = participants[["participant_id", "UI", "숙련도"]].copy()
outcomes["결과"] = np.where(
    outcomes["participant_id"].isin(failure_ids),
    "Failure", "Success",
)
display(outcomes.head())


### 7-4. 성공률 표 함수 만들기


In [ ]:
def success_table(data, group_columns):
    count = (
        data.groupby(group_columns + ["결과"], observed=True)
        .size().unstack(fill_value=0)
        .reindex(columns=["Success", "Failure"], fill_value=0)
    )
    total = count.sum(axis=1)
    result = pd.DataFrame(index=count.index)
    labels = {
        "Success": "두 과제 모두 완료 n (%)",
        "Failure": "한 과제 이상 미완료 n (%)",
    }
    for outcome in ["Success", "Failure"]:
        result[labels[outcome]] = [
            f"{n} ({n / t:.1%})" for n, t in zip(count[outcome], total)
        ]
    result["전체"] = total
    return result


### 7-5. UI별 성공률 표시하기


In [ ]:
ui_success = success_table(outcomes, ["UI"])
overall_success = success_table(
    outcomes.assign(전체="전체"), ["전체"]
)
display(pd.concat([ui_success, overall_success]))


### 7-6. UI와 성공 여부의 관련성 검정하기

실패 사례가 적으므로 카이제곱검정보다 Fisher 정확검정을 사용합니다.


In [ ]:
success_counts = pd.crosstab(
    outcomes["UI"], outcomes["결과"]
).reindex(
    index=["A2UI", "Baseline"],
    columns=["Success", "Failure"],
    fill_value=0,
)
odds_ratio, fisher_p = stats.fisher_exact(success_counts)

fisher_result = pd.DataFrame({
    "검정": ["Fisher 정확검정"],
    "오즈비": [odds_ratio],
    "p": [fisher_p],
}).set_index("검정")
display(fisher_result.round(4))


**완료율 해석:** 위 표의 UI별 완료율과 Fisher 정확검정 결과를 확인합니다. 유의하지 않은 결과는 두 UI가 동일하다는 증명이 아닙니다.

## 8. 최종 요약

1. 40명의 참가자에게서 Task 1과 Task 2 로그를 각각 집계해 80개의 참가자 × Task 관측치를 만들었습니다.
2. Number of Prompts는 실제 질문 행만 집계했고, 수행시간은 각 Task의 실제 로그 구간을 분 단위로 계산했습니다.
3. 두 종속변수에 각각 `UI × 숙련도 × Task` 3요인 모형을 적용했으며 과제 순서는 포함하지 않았습니다.
4. 두 분석 모두 7개 효과 중 Task 주효과만 유의했습니다: Number of Prompts F(1, 39)=19.361, p<.001; 수행시간 F(1, 39)=32.246, p<.001.
5. UI 관련 효과는 유의하지 않았고, 참가자 단위 성공률에서도 UI 관련성을 확인하지 못했습니다.

따라서 현재 자료에서는 **Task 2가 Task 1보다 더 많은 프롬프트와 수행시간을 요구했다**는 결론이 가장 분명합니다. 반면 A2UI가 Baseline보다 프롬프트 사용량, 수행시간 또는 전체 과제 완료율을 개선했다고 단정할 근거는 확인되지 않았습니다.
